In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torchaudio
import torch
import librosa
import parselmouth
import maad
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score
from scipy.stats import skew, kurtosis
from scipy.signal import hilbert
from spafe.features.lfcc import lfcc
from maad.features import temporal_entropy, all_temporal_features

Load FIles with torch

In [2]:
csvTotale = pd.read_csv('.././data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvTotale['gender'] = csvTotale['gender'].map(lambda x: 0 if x=='male' else 1)
csvTotale['tempo'] = csvTotale['tempo'].map(lambda x: float(x[1:-1]))
csvTotale['path'] = csvTotale['path'].map(lambda x: x.split('/')[-1])

In [23]:
audioDevelopment = {file:torchaudio.transforms.Resample(orig_freq=44100, new_freq=16000)(torchaudio.load('.././data/audios_development/'+file)[0]) for file in csvTotale['path']}

In [57]:
def getMFCC(audio, imp=0):  # Keep imp=1 otherwise the score worsens
    numFcc = 35
    mfccs = librosa.feature.mfcc(y=audio, sr=22050, n_mfcc=numFcc)
    if imp:
        energy = np.sum(librosa.amplitude_to_db(np.abs(librosa.stft(audio))**2, ref=np.max))
        
        deltas = librosa.feature.delta(mfccs).sum(axis=1)
        double_deltas = librosa.feature.delta(mfccs, order=2).sum(axis=1)

        return pd.Series(np.concatenate((mfccs.mean(axis=1), deltas, double_deltas, [energy]), axis=0),
                            index=[f'mfcc_{i}' for i in range(numFcc)]+['delta_'+str(i) for i in range(numFcc)]+['double_delta_'+str(i) for i in range(numFcc)]+['energy'])
    else:
        return pd.Series(mfccs.mean(axis=1), index=[f'mfcc_mean{i}' for i in range(numFcc)])

In [58]:
def computeMath(audio):
    return pd.Series([skew(audio), kurtosis(audio), np.mean(np.abs(hilbert(audio))), np.mean(np.abs(np.fft.fft(audio)))],
                     index=['skew', 'kurtosis', 'hilbertMean', 'fftMean'])

In [59]:
def computeStats(audio):
    sound = parselmouth.Sound(audio)
    pitch = sound.to_pitch()
    info = str(pitch.info()).split('\n')
    return pd.Series([pitch.count_voiced_frames(), pitch.get_mean_absolute_slope(), pitch.xmax-pitch.xmin, 
                      pitch.n_frames, *[float(info[15+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 5)],
                      *[float(info[21+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 3)]],
                     index=['nVoicedFrames', 'meanAbsoluteSlope', 'duration', 'nFrames', 
                            'q10', 'q16', 'q50', 'q84', 'q90', '84-median', 'median-16', '90-10']
                     )

In [60]:
def getSpectralEnergy(audio):
    S = librosa.stft(audio)
    freqs = librosa.fft_frequencies(sr=22050)
    return pd.Series([np.sum(np.abs(S[(freqs >= 150) & (freqs <= 650)])**2), np.sum(np.abs(S[(freqs >= 1000) & (freqs <= 8000)])**2)],
                        index=['spectralEnergy250-650', 'spectralEnergy1000-8000'])

In [61]:
def getLFCC(audio):
    numcc = 35
    return pd.Series(lfcc(audio, fs=22050, num_ceps=numcc, low_freq=50, high_freq=3000, nfft=256, nfilts=numcc+1).mean(axis=0),
                        index=[f'lfcc{i}' for i in range(numcc)])

In [62]:
def computeEntropy(audio):
    return pd.Series(temporal_entropy(audio), index=['entropy'])

In [63]:
def computeAllFeatures(audio):
    return pd.Series(all_temporal_features(audio, fs=22050, nperseg=256).values[0],
                     index=['sm', 'sv', 'ss', 'sk', 'Time 5%', "Time 25%", "Time 50%", "Time 75%", "Time 95%  ", 
                            "zcr", "duration_5", "duration_90"])

In [64]:
def computeMetrics(audios):
    return pd.DataFrame({file: pd.concat([
            getMFCC((temp:=audios[file][0].numpy())),
            getSpectralEnergy(temp),
            computeMath(temp),
            computeAllFeatures(temp),
            computeEntropy(temp),   # fucking entropy drops the score
            getLFCC(temp),
            computeStats(temp)
            ], axis=0)
        for file in audios}).T

In [65]:
metrics = computeMetrics(audioDevelopment)

In [67]:
cross_val_score(HistGradientBoostingRegressor(), metrics,
                 csvTotale['age'], cv=20, scoring='neg_mean_absolute_error').mean()

np.float64(-6.900108019011472)

In [68]:
csvEval = pd.read_csv('.././data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvEval['gender'] = csvEval['gender'].map(lambda x: 0 if x=='male' else 1)
csvEval['tempo'] = csvEval['tempo'].map(lambda x: float(x[1:-1]))
csvEval['path'] = csvEval['path'].map(lambda x: x.split('/')[-1])

In [69]:
audioEval = {file:torchaudio.transforms.Resample(orig_freq=44100, new_freq=16000)(torchaudio.load('.././data/audios_evaluation/'+file)[0]) for file in csvEval['path']}

In [70]:
metricsEval = computeMetrics(audioEval)

In [71]:
ypred = pd.Series(HistGradientBoostingRegressor().fit(metrics, csvTotale['age'])
          .predict(metricsEval),
          name='Predicted', index=csvEval.index)

In [74]:
from sklearn.metrics import root_mean_squared_error
pd.DataFrame({"nowModel":[
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9650.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9308.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('submissionOggi.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predictedNoHIST-9426.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9322.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9235.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9555.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9196.csv', header=0, index_col=0)['Predicted'], ypred)
    ],
 'bestModel':[0,
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9650.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('predicted-9308.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('submissionOggi.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predictedNoHIST-9426.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('predicted-9322.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9235.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9555.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9196.csv', header=0, index_col=0)['Predicted'])
     ],
 })

,nowModel,bestModel
0,4.556606,0.000000
1,4.693383,3.246054
2,4.483056,2.344467
3,4.457511,3.056776
4,4.355893,3.268457
5,4.688768,3.378994
6,4.609429,3.284294
7,5.328392,4.904805
8,4.629234,3.094170
